# futures_4h_jma_atr_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (81).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `futures` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 8 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, fast_backtest, backtest_trades, calculate_metrics, calculate_composite_score, run_backtest, evaluate_params, walk_forward_optimization |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Comodoties_Data/Comodoties_Data_merged.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2021-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['BRENTCMDUSD60']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple
from numba import njit

# ===============================
# ATR Calculator
# ===============================
class ATRIndicator:
    """Average True Range (ATR) Calculator"""
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high = df[self.high_col]
        low = df[self.low_col]
        close = df[self.close_col]

        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)

        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)  # shift to prevent lookahead

# ===============================
# JMA + Signals (manual loop)
# ===============================
def compute_jma_and_signals(df, smooth_length=14, jma_normalization_period=60,
                            upper_threshold=1.25, lower_threshold=-0.75,
                            rs_norm_period=40, rs_period=35, rs_threshold=0.35):
    df = df.copy()
    price = df['close'].shift(1)

    # --- JMA calculation (manual loop) ---
    alpha = 2.0 / (smooth_length + 1)
    jma_values = price.ewm(alpha=alpha, adjust=False).mean()
    df['JMA'] = jma_values # Assign JMA values to a new column

    # --- Normalization ---
    df['Normalized_JMA'] = ((df['JMA'] - df['JMA'].rolling(jma_normalization_period).mean()) / \
                            df['JMA'].rolling(jma_normalization_period).std())

    # --- Rogers-Satchell Volatility ---
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    log_hc = np.log(h / c)
    log_ho = np.log(h / o)
    log_lc = np.log(l / c)
    log_lo = np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6
    df['RS_Norm'] = df['RS_Volatility'].rolling(rs_norm_period).apply(lambda x: x.rank(pct=True).iloc[-1])

    # --- Signals ---
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Norm'] > rs_threshold), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Norm'] > rs_threshold), 'Signal'] = 2

    return df

# ===============================
# ATR_max computation helper
# ===============================
def compute_atr_max(group, atr_mult):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * atr_mult,
        group['ATR_5'].rolling(20).max() * atr_mult
    ], axis=1).max(axis=1)

# ===============================
# Full Data Preparation
# ===============================
# Assuming 'data' DataFrame exists with required columns
data['Date'] = pd.to_datetime(data['Date'])

# Apply JMA + Signals per ticker
jma_results = []
for ticker, group in data.groupby('Ticker'):
    jma_group = compute_jma_and_signals(group)
    jma_results.append(jma_group)
data = pd.concat(jma_results, ignore_index=True)

# Initialize ATR calculator
atr_calc = ATRIndicator()

# Compute ATRs per ticker with shift=1
for period in [14, 5]:
    col_name = f'ATR_{period}'
    atr_results = []
    for ticker, group in data.groupby('Ticker'):
        atr_vals = atr_calc.compute_atr(group, period=period, shift=1)
        atr_results.append(atr_vals)
    data[col_name] = pd.concat(atr_results, ignore_index=True)

# Compute ATR_max
atr_max_results = []
for ticker, group in data.groupby('Ticker'):
    atr_max_vals = compute_atr_max(group, atr_mult=0.5)
    atr_max_results.append(atr_max_vals)
data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)
data['ATR_max'] = data['ATR_max']

# Shift signals for next-bar execution
data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(np.int8)


@njit
def fast_backtest(prices, signals, atr_max, capital, fee_rate,
                  slippage_rate, entry_sl_pct):
    """Numba-optimized backtest - 10-100x faster"""

    n = len(prices)
    max_trades = n  # Pre-allocate
    trades = np.empty((max_trades, 8), dtype=np.float64)
    trade_idx = 0

    cumulative_pnl = 0.0
    position_open = False
    entry_idx = last_exit_idx = -1
    entry_price = sl_price = quantity = direction = 0.0
    prev_close = prev_atr = 0.0

    for i in range(n):
        open_, high, low, close = prices[i]
        signal, atr = signals[i], atr_max[i]

        # === ENTRY ===
        if not position_open and i != last_exit_idx and signal in (1, 2):
            direction = 1.0 if signal == 1 else -1.0
            entry_price = open_ * (1 + slippage_rate * direction)
            quantity = capital / entry_price
            entry_idx = i
            position_open = True

            sl_price = entry_price * (1 - entry_sl_pct * direction)
            entry_sl_hit = (low <= sl_price if direction == 1 else high >= sl_price)

            if entry_sl_hit:
                exit_price = sl_price * (1 - slippage_rate * direction)
                pnl = (exit_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl

                trades[trade_idx] = [entry_idx, i, entry_price, exit_price,
                                    net_pnl, cumulative_pnl, direction, 0]
                trade_idx += 1
                position_open = False
                last_exit_idx = i

            prev_close, prev_atr = close, atr
            continue

        # === POSITION MANAGEMENT ===
        if position_open and i != entry_idx:
            new_sl = prev_close - prev_atr * direction
            sl_price = max(sl_price, new_sl) if direction == 1 else min(sl_price, new_sl)
            sl_hit = (low <= sl_price if direction == 1 else high >= sl_price)

            if sl_hit:
                exit_price = sl_price * (1 - slippage_rate * direction)
                pnl = (exit_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl

                trades[trade_idx] = [entry_idx, i, entry_price, exit_price,
                                    net_pnl, cumulative_pnl, direction, 1]
                trade_idx += 1
                position_open = False
                last_exit_idx = i

        prev_close, prev_atr = close, atr

    return trades[:trade_idx]


def backtest_trades(data: pd.DataFrame, entry_sl_pct: float = 0.02,
                   initial_capital: float = 100000, fee_rate: float = 0.0005,
                   slippage_rate: float = 0.00025) -> pd.DataFrame:
    """Vectorized multi-ticker backtest"""

    required_cols = ['Date', 'Ticker', 'open', 'high', 'low', 'close', 'Signal', 'ATR_max']
    data = data[required_cols].copy()
    data['Date'] = pd.to_datetime(data['Date'])

    all_trades = []

    for ticker, group in data.groupby('Ticker', sort=False):
        prices = group[['open', 'high', 'low', 'close']].values
        signals = group['Signal'].values
        atr_max = group['ATR_max'].values
        dates = group['Date'].values

        trades = fast_backtest(prices, signals, atr_max, initial_capital,
                              fee_rate, slippage_rate, entry_sl_pct)

        if len(trades) > 0:
            df = pd.DataFrame(trades, columns=[
                'entry_idx', 'exit_idx', 'Entry Price', 'Exit Price',
                'PnL', 'Cumulative PnL', 'direction', 'exit_type'
            ])

            df['Ticker'] = ticker
            df['Entry Time'] = dates[df['entry_idx'].astype(int)]
            df['Exit Time'] = dates[df['exit_idx'].astype(int)]
            df['Direction'] = df['direction'].map({1.0: 'long', -1.0: 'short'})
            df['Exit Reason'] = df['exit_type'].map({0: 'Entry SL', 1: 'Trailing SL'})

            df = df[['Ticker', 'Entry Time', 'Exit Time', 'Entry Price',
                    'Exit Price', 'PnL', 'Exit Reason', 'Cumulative PnL', 'Direction']]
            all_trades.append(df)

    return pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()


# === Example Usage ===
tradebook = backtest_trades(
    data,
    entry_sl_pct=0.01,
    initial_capital=100000,
    fee_rate=0.0005,
    slippage_rate=0.00025
)

tradebook.to_csv("tradebook_BRENTCMDUSD60_Q7.csv", index=False)
tradebook

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from itertools import product
from typing import Dict
from sklearn.metrics import r2_score
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import warnings
warnings.filterwarnings('ignore')

# ===============================
# CORE METRICS
# ===============================

def calculate_metrics(trades, initial_capital=100000):
    """Calculate all performance metrics from trades."""
    if len(trades) == 0:
        return {k: 0 for k in ['total_trades', 'profit_to_drawdown', 'equity_r2',
                                'monthly_consistency', 'daily_drawdown', 'total_pnl',
                                'win_rate', 'avg_pnl_per_trade', 'max_drawdown']}

    pnl = trades['PnL'].values
    total_pnl = pnl.sum()
    n_trades = len(trades)

    # Sort and compute cumulative PnL
    trades = trades.sort_values('Exit Time').reset_index(drop=True)
    cum_pnl = pnl.cumsum() if len(pnl) != len(trades) else trades['PnL'].cumsum().values

    # Drawdown metrics
    running_max = np.maximum.accumulate(cum_pnl)
    drawdown = running_max - cum_pnl
    max_dd = drawdown.max()
    profit_to_dd = total_pnl / max_dd if max_dd > 0 else 0

    # Equity R²
    equity_r2 = 0
    if n_trades >= 2:
        try:
            x = np.arange(n_trades)
            fit = np.polyfit(x, cum_pnl, 1)
            equity_r2 = max(0, r2_score(cum_pnl, np.polyval(fit, x)))
        except:
            pass

    # Time-based metrics
    exit_times = pd.to_datetime(trades['Exit Time'])

    # Monthly consistency
    monthly_pnl = trades.groupby(exit_times.dt.to_period('M'))['PnL'].sum()
    monthly_consistency = (monthly_pnl > 0).sum() / len(monthly_pnl) if len(monthly_pnl) >= 2 else 0

    # Daily drawdown
    daily_pnl = trades.groupby(exit_times.dt.date)['PnL'].sum().values
    daily_cum = np.cumsum(daily_pnl)
    daily_dd = np.maximum.accumulate(daily_cum) - daily_cum
    daily_dd_score = 1 - min(1, daily_dd.mean() / initial_capital)

    return {
        'total_trades': n_trades,
        'profit_to_drawdown': profit_to_dd,
        'equity_r2': equity_r2,
        'monthly_consistency': monthly_consistency,
        'daily_drawdown': daily_dd_score,
        'total_pnl': total_pnl,
        'win_rate': (pnl > 0).sum() / n_trades,
        'avg_pnl_per_trade': total_pnl / n_trades,
        'max_drawdown': max_dd
    }


def calculate_composite_score(metrics, weights, all_pd=None):
    """Calculate weighted composite score."""
    pd_norm = metrics['profit_to_drawdown']
    if all_pd and len(all_pd) > 0:
        max_val = np.percentile(all_pd, 95)
        pd_norm = min(1.0, pd_norm / max_val) if max_val > 0 else 0
    else:
        pd_norm = min(1.0, pd_norm / 5.0)

    return (weights['profit_to_drawdown'] * pd_norm +
            weights['equity_r2'] * metrics['equity_r2'] +
            weights['monthly_consistency'] * metrics['monthly_consistency'] +
            weights['daily_drawdown'] * metrics['daily_drawdown'])


# ===============================
# BACKTEST ENGINE
# ===============================

def run_backtest(data, params, initial_capital, fee_rate, slippage_rate, entry_sl_pct):
    """Run backtest with given parameters."""
    try:
        # Prepare data
        prepared = data.copy()

        # Apply JMA signals per ticker
        jma_list = []
        for ticker, group in prepared.groupby('Ticker'):
            jma_group = compute_jma_and_signals(
                group,
                jma_normalization_period=params.get('jma_normalization_period', 60),
                upper_threshold=params.get('upper_threshold', 1.25),
                lower_threshold=params.get('lower_threshold', -0.75)
            )
            jma_list.append(jma_group)
        prepared = pd.concat(jma_list, ignore_index=True)

        # Compute ATR indicators
        atr_calc = ATRIndicator()
        for period in [14, 5]:
            atr_results = []
            for ticker, group in prepared.groupby('Ticker'):
                atr_results.append(atr_calc.compute_atr(group, period=period, shift=1))
            prepared[f'ATR_{period}'] = pd.concat(atr_results, ignore_index=True)

        # Compute ATR_max
        atr_max_list = []
        for ticker, group in prepared.groupby('Ticker'):
            atr_max_list.append(compute_atr_max(group, atr_mult=0.1))
        prepared['ATR_max'] = pd.concat(atr_max_list, ignore_index=True)

        # Shift signals for next-bar execution
        prepared['Signal'] = prepared.groupby('Ticker', group_keys=False)['Signal'].shift(1)
        prepared = prepared.dropna(subset=['Signal'])
        prepared['Signal'] = prepared['Signal'].astype(np.int8)

        # Execute backtest
        return backtest_trades(prepared, entry_sl_pct, initial_capital, fee_rate, slippage_rate)
    except:
        return None


def evaluate_params(args):
    """Evaluate single parameter set (parallel worker)."""
    params, data, initial_capital, fee_rate, slippage_rate, entry_sl_pct = args
    trades = run_backtest(data, params, initial_capital, fee_rate, slippage_rate, entry_sl_pct)
    if trades is None or len(trades) == 0:
        return None, None
    return params, calculate_metrics(trades, initial_capital)


# ===============================
# WALK FORWARD OPTIMIZATION
# ===============================

def walk_forward_optimization(
    data: pd.DataFrame,
    param_grid: Dict,
    in_sample_months: int = 6,
    out_sample_months: int = 3,
    step_months: int = 3,
    initial_capital: float = 100000,
    fee_rate: float = 0.0005,
    slippage_rate: float = 0.00025,
    entry_sl_pct: float = 0.02,
    composite_weights: Dict = None,
    n_jobs: int = -1
):
    """
    Streamlined Walk Forward Optimization with parallel processing.
    Optimizes using composite score of multiple metrics.
    """
    # Default weights
    weights = composite_weights or {
        'profit_to_drawdown': 0.25,
        'equity_r2': 0.25,
        'monthly_consistency': 0.25,
        'daily_drawdown': 0.25
    }

    # Setup
    n_jobs = max(1, mp.cpu_count() - 1) if n_jobs == -1 else n_jobs
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    # Generate parameter combinations
    param_combos = list(product(*param_grid.values()))
    param_names = list(param_grid.keys())

    # Generate windows
    min_date, max_date = data['Date'].min(), data['Date'].max()
    windows = []
    current = min_date

    while current + pd.DateOffset(months=in_sample_months + out_sample_months) <= max_date:
        train_end = current + pd.DateOffset(months=in_sample_months)
        test_end = train_end + pd.DateOffset(months=out_sample_months)
        windows.append({
            'train': (current, train_end),
            'test': (train_end, test_end)
        })
        current += pd.DateOffset(months=step_months)

    # Optimization loop
    results = []
    all_trades = []

    pbar = tqdm(total=len(windows), desc="Walk Forward Optimization")

    for idx, window in enumerate(windows):
        # Split data
        train_data = data[(data['Date'] >= window['train'][0]) & (data['Date'] < window['train'][1])].copy()
        test_data = data[(data['Date'] >= window['test'][0]) & (data['Date'] < window['test'][1])].copy()

        if len(train_data) == 0 or len(test_data) == 0:
            pbar.update(1)
            continue

        # Prepare evaluation arguments
        param_dicts = [dict(zip(param_names, combo)) for combo in param_combos]
        args = [(p, train_data, initial_capital, fee_rate, slippage_rate, entry_sl_pct) for p in param_dicts]

        # Parallel optimization
        best_params, best_score, best_metrics = None, -np.inf, None
        metrics_list = []

        with ProcessPoolExecutor(max_workers=n_jobs) as executor:
            for params, metrics in executor.map(evaluate_params, args):
                if metrics is None:
                    continue
                metrics_list.append(metrics)
                score = calculate_composite_score(metrics, weights, [m['profit_to_drawdown'] for m in metrics_list])
                if score > best_score:
                    best_score, best_params, best_metrics = score, params, metrics

        if best_params is None:
            pbar.update(1)
            continue

        # Test best parameters
        test_trades = run_backtest(test_data, best_params, initial_capital, fee_rate, slippage_rate, entry_sl_pct)

        if test_trades is not None and len(test_trades) > 0:
            test_metrics = calculate_metrics(test_trades, initial_capital)

            # Store results
            results.append({
                'window': idx + 1,
                'train_start': window['train'][0],
                'train_end': window['train'][1],
                'test_start': window['test'][0],
                'test_end': window['test'][1],
                **{f'best_{k}': v for k, v in best_params.items()},
                **{f'train_{k}': v for k, v in best_metrics.items()},
                **{f'test_{k}': v for k, v in test_metrics.items()}
            })

            test_trades['window'] = idx + 1
            all_trades.append(test_trades)

        pbar.update(1)
        pbar.set_postfix({'Score': f'{best_score:.3f}', 'Test P/DD': f'{test_metrics["profit_to_drawdown"]:.2f}' if test_trades is not None and len(test_trades) > 0 else 'N/A'})

    pbar.close()

    # Combine results
    results_df = pd.DataFrame(results)
    trades_df = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()

    return results_df, trades_df


# ===============================
# USAGE EXAMPLE
# ===============================

param_grid = {
    'jma_normalization_period': [30, 60, 90, 120, 150, 180],
    'upper_threshold': [1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5],
    'lower_threshold': [-0.75, -1.0, -1.25, -1.5, -1.75, -2.0, -2.25, -2.5]
}

composite_weights = {
    'profit_to_drawdown': 0.30,
    'equity_r2': 0.30,
    'monthly_consistency': 0.20,
    'daily_drawdown': 0.20
}

# Run optimization
wfo_results, oos_trades = walk_forward_optimization(
    data=data,
    param_grid=param_grid,
    in_sample_months=12,
    out_sample_months=3,
    step_months=1,
    initial_capital=100000,
    fee_rate=0.0005,
    slippage_rate=0.00025,
    entry_sl_pct=0.02,
    composite_weights=composite_weights,
    n_jobs=-1
)

# Save results
wfo_results.to_csv("wfo_results.csv", index=False)
oos_trades.to_csv("wfo_trades.csv", index=False)

# Summary
print("\n" + "="*80)
print("WALK FORWARD OPTIMIZATION COMPLETED")
print("="*80)
print(f"Windows: {len(wfo_results)} | Total OOS Trades: {len(oos_trades)}")

if len(oos_trades) > 0:
    m = calculate_metrics(oos_trades, 100000)
    print("\n" + "="*80)
    print("AGGREGATE OUT-OF-SAMPLE PERFORMANCE")
    print("="*80)
    print(f"Total PnL: ${m['total_pnl']:,.2f}")
    print(f"Win Rate: {m['win_rate']:.2%}")
    print(f"Profit/DD: {m['profit_to_drawdown']:.2f}")
    print(f"Equity R²: {m['equity_r2']:.3f}")
    print(f"Monthly Consistency: {m['monthly_consistency']:.2%}")
    print(f"Daily DD Score: {m['daily_drawdown']:.3f}")
    print(f"Max DD: ${m['max_drawdown']:,.2f}")
    print(f"Avg PnL/Trade: ${m['avg_pnl_per_trade']:,.2f}")

    print("\n" + "="*80)
    print("AVERAGE TEST METRICS ACROSS WINDOWS")
    print("="*80)
    print(f"Avg Profit/DD: {wfo_results['test_profit_to_drawdown'].mean():.2f}")
    print(f"Avg Equity R²: {wfo_results['test_equity_r2'].mean():.3f}")
    print(f"Avg Monthly Consistency: {wfo_results['test_monthly_consistency'].mean():.2%}")
    print(f"Avg Daily DD Score: {wfo_results['test_daily_drawdown'].mean():.3f}")
    print("="*80)